## **ENCODING AN ARBITRARY HAMILTONIAN IN THE BASIS OF PAULI MATRICES**

In [1]:
!pip install qiskit --quiet
!pip install qiskit_ibm_runtime --quiet

import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as LA
from qiskit import *
from qiskit.quantum_info import Operator
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler, Batch

In [2]:
#pauli matrix decomposition - utility functions
def vec_query(arr, my_dict):
    import numpy as np
    return np.vectorize(my_dict.__getitem__, otypes=[tuple])(arr)

def nested_kronecker_product(a):
    import numpy as np
    if len(a) == 2:
        return np.kron(a[0], a[1])
    else:
        return np.kron(a[0], nested_kronecker_product(a[1:]))
    
def Hilbert_Schmidt(mat1, mat2):
    import numpy as np
    return np.trace(mat1.conj().T*mat2)

In [3]:
#pauli matrix decomposition
def decompose(Ham_arr, tol=10):
    import numpy as np
    import itertools

    #define a dictionary with the four pauli matrices
    pms = {
        'I': np.array([[1, 0], [0, 1]], dtype=complex), 
        'X': np.array([[0, 1], [1, 0]], dtype=complex), 
        'Y': np.array([[0, -1j], [1j, 0]], dtype=complex), 
        'Z': np.array([[1, 0], [0, -1]], dtype=complex)
    }
    pauli_keys = list(pms.keys())   #keys of the dictionary, i.e. 'I', 'X', 'Y', 'Z'
    nqb = int(np.log2(Ham_arr.shape[0]))  #number of qubits
    output_string = '' #initialize output string

    #make all possible combinations of the pauli matrices for nqb qubits
    sigma_combinations = list(itertools.product(pauli_keys, repeat=nqb))

    #loop through all combinations, calculate the kronecker product, and calculate the coefficient using the Hilbert-Schmidt inner product
    for ii in range(len(sigma_combinations[ii])):
        #take the full pauli string
        pauli_str = ''.join(sigma_combinations[ii])
        # Compute the coefficient for each Pauli string. This is done as follows:
        # 1) Evaluate the tensor product of Pauli String to get a 2^n by 2^n matrix:
        # Turn each element of Pauli String ('ZIXI') into its corresponding
        # 2 by 2 Pauli Matrix, then evaluate the Kronecker product.
        # 2) Compute the Hilbert-Schmidt Inner Product.
        # 3) Normalize by (1/(2^n)).
    norm_factor = 1/(2**nqb)

    #convert the pauli string to list of arrays
    tmp_mat_list = vec_query(np.array(sigma_combinations[ii]), pms)
    
    #evaluate the kronecker product
    tmp_p_matrix = nested_kronecker_product(tmp_mat_list)
    hs_innerprod = Hilbert_Schmidt(tmp_p_matrix, Ham_arr)
    a_coeff = norm_factor * hs_innerprod

    #if the coefficient is non-zero
    if a_coeff != 0.0:
        #assert that cooeficient is less than the tolerance
        min_val = 10**(-tol)
        if abs(a_coeff) < min_val:
            pass
        #if non-zero
        else:
            output_string += str(np.round(a_coeff.real, tol))+"*"+ alt_name
            output_string += '+' # Add a plus sign for the next term!

    return output_string[:-1] # To ignore that extra plus sign
